# EDA — Harvester Risk Atlas (EthnoHACK 2026 Track 3)

Analyst notebook: occurrence coverage, HPI component distributions, spatial summary.
**Not medical advice.** IUCN statuses are never invented.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["axes.facecolor"] = "#ffffff"
plt.rcParams["figure.facecolor"] = "#f5f5f7"
plt.rcParams["axes.edgecolor"] = "#d2d2d7"
ACCENT = "#2d6a4f"

In [ ]:
hpi = pd.read_csv(ROOT / "data/processed/hpi_scores.csv")
occ_path = ROOT / "data/processed/occurrences_sample.parquet"
occ = pd.read_parquet(occ_path) if occ_path.exists() else pd.DataFrame()
print(hpi.shape, "species")
hpi[["scientific_name", "hpi", "hpi_band", "n_occurrences"]].sort_values("hpi", ascending=False).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(hpi["hpi"], bins=12, color=ACCENT, edgecolor="#d2d2d7", ax=ax)
ax.set_xlabel("HPI v1")
ax.set_ylabel("Species count")
ax.set_title("HPI distribution")
plt.tight_layout()
plt.show()

In [ ]:
comps = hpi[["component_rarity", "component_climate", "component_harvest", "component_pa_gap"]]
fig, ax = plt.subplots(figsize=(7, 3.8))
comps.boxplot(ax=ax, color=dict(boxes=ACCENT, whiskers="#6e6e73", medians="#1d1d1f", caps="#6e6e73"))
ax.set_ylabel("Score [0,1]")
ax.set_title("HPI component spreads")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
comps.describe().round(3)

In [ ]:
if not occ.empty:
    cov = occ.groupby("scientific_name").size().rename("n").reset_index()
    fig, ax = plt.subplots(figsize=(8, 5))
    cov.sort_values("n").plot.barh(x="scientific_name", y="n", ax=ax, color=ACCENT, legend=False)
    ax.set_xlabel("GBIF sample points")
    ax.set_title("Occurrence coverage (demo extract)")
    plt.tight_layout()
    plt.show()
else:
    print("No occurrence parquet — run scripts/fetch_gbif.py")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = hpi[["component_rarity", "component_climate", "component_harvest", "component_pa_gap"]].fillna(0.5)
Z = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(Z)
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=hpi["hpi"], cmap="YlOrRd", edgecolors="#d2d2d7", s=60)
plt.colorbar(sc, ax=ax, label="HPI")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
ax.set_title("Species in HPI component space (PCA)")
plt.tight_layout()
plt.show()

## Notes for judging
- Climate anomaly is a **labeled proxy** until CHELSA/WorldClim rasters are attached.
- PA gap uses indicative centroid buffers — upgrade to WDPA.
- Confidence column flags imputation; do not oversell precision.